# Sales Service load test on Kaggle CPU

Create a new Kaggle Notebook (Python, CPU only), enable **Internet** in Session options, then import this notebook and run all cells. It clones the public repository at its current main commit, installs MySQL 8 and Java 21, starts both services locally, runs Locust, and writes CSV/HTML/Markdown evidence under `/kaggle/working/sales_load`. This is an isolated benchmark: never point it at your personal MySQL database.

The first setup may take several minutes. If package installation is blocked, stop and capture the error rather than switching to H2 or claiming MySQL results. The script refuses to run against an existing `sales_service` schema.


In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/sales-service')
if repo.exists():
    raise RuntimeError('Repository already exists: start a fresh Kaggle session for an independent run')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/khoizz247/sales-service.git', str(repo)], check=True)
print('Git commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


The next cell runs four phases: smoke (5 users, 1 minute), baseline (10 users, 2 minutes), main (30 users, 5 minutes), and high (60 users, 5 minutes if the main phase has no failed requests). For a short feasibility check first, uncomment `env['LOAD_SMOKE_ONLY'] = '1'`, then use a fresh session for the full run.


In [ ]:
import os
import sys

env = os.environ.copy()
# env['LOAD_SMOKE_ONLY'] = '1'
subprocess.run([sys.executable, str(repo / 'performance/kaggle_run.py')], cwd=repo, env=env, check=True)


In [ ]:
runs = sorted(Path('/kaggle/working/sales_load').glob('*'))
latest = runs[-1]
print('Evidence:', latest)
print((latest / 'REPORT.md').read_text(encoding='utf-8'))
print('Files:', *sorted(p.name for p in latest.iterdir()), sep='\n- ')
import shutil
archive = shutil.make_archive('/kaggle/working/sales_load_results', 'zip', root_dir='/kaggle/working', base_dir='sales_load')
print('Download this ZIP from Kaggle Output:', archive)
